# Full ETL Rootzone Pipeline

This notebook creates a rootzone `master.csv`-format file from Bet Dagan weather/radiation data, then runs the final no-RH 48H rootzone model when pH/EC anchor and fert/irrigation inputs are available.

In [10]:
# ============================================================
# EDIT ONLY THIS CELL
# ============================================================

# Bet Dagan input files. These defaults point to the project raw-data folder.
# You can also put new files in this saved_model_no_rh folder and use only the file names.
WEATHER_FILE = '../../data/raw/bet_dagan_weather.csv'
RADIATION_FILE = '../../data/raw/bet_dagan_radiation.csv'

# Saved micro-climate model. None means use micro_climate_3day_unified_model.joblib from this folder.
MICRO_MODEL_FILE = None

# Optional existing filled master file.
# Use 'master.csv' for the current project data so pH/EC, fert/irrigation, canopy, and DAP are preserved.
# Use None for new data, run once, fill etl_master_template.csv, then set this to 'etl_master_template.csv'.
MANUAL_MASTER_FILE = 'master.csv'

# Output files.
OUTPUT_MASTER_FILE = 'etl_master_template.csv'
OUTPUT_FORECAST_FILE = 'etl_micro_climate_predictions.csv'
OUTPUT_PREDICTION_FILE = 'etl_rootzone_prediction.csv'

# Optional time range. None means use the overlap between the weather and radiation files.
START_TIME = '2025-08-10 07:40'
END_TIME = '2025-09-11 07:30'

# Crop metadata used when MANUAL_MASTER_FILE is None or lacks these columns.
# Current project planting date was 2025-05-29.
PLANTING_DATE = '2025-05-29'
CANOPY_COVER_DEFAULT = 0.0

# Rootzone prediction controls.
# If False, the notebook only creates the editable master-format CSV.
RUN_ROOTZONE_PREDICTION = True

# None means use the last timestamp in the generated master.
# For the current example, this matches the existing predict_rootzone notebook default target.
TARGET_TIME = '2025-09-11 07:30'

# None means use the latest row before TARGET_TIME with both ph and ec_ms filled.
ANCHOR_TIME = '2025-09-10 07:40'

# Model constraints.
REQUIRED_HISTORY_HOURS = 48.0
DARK_6H_WINDOW_HOURS = 6.0
MAX_EXTERNAL_GAP_HOURS = 3.0
# ============================================================


In [11]:
from pathlib import Path
import importlib.util
import sys

REQUIRED_PACKAGES = [
    ('numpy', 'numpy'),
    ('pandas', 'pandas'),
    ('joblib', 'joblib'),
    ('sklearn', 'scikit-learn'),
    ('xgboost', 'xgboost'),
    ('lightgbm', 'lightgbm'),
]
missing_packages = [install_name for import_name, install_name in REQUIRED_PACKAGES if importlib.util.find_spec(import_name) is None]
if missing_packages:
    raise ModuleNotFoundError(
        'Missing Python packages needed for the full ETL pipeline: '
        + ', '.join(missing_packages)
        + '. Install them before running this notebook.'
    )

for candidate in [Path.cwd(), Path.cwd() / 'scripts' / 'saved_model_no_rh', Path.cwd().parent / 'saved_model_no_rh']:
    if (candidate / 'rootzone_full_etl.py').exists():
        sys.path.insert(0, str(candidate))
        break

from rootzone_full_etl import run_pipeline, print_pipeline_result

print('Full ETL setup ready')


Full ETL setup ready


In [12]:
result = run_pipeline(
    weather_file=WEATHER_FILE,
    radiation_file=RADIATION_FILE,
    micro_model_file=MICRO_MODEL_FILE,
    manual_master_file=MANUAL_MASTER_FILE,
    output_master_file=OUTPUT_MASTER_FILE,
    output_forecast_file=OUTPUT_FORECAST_FILE,
    output_prediction_file=OUTPUT_PREDICTION_FILE,
    start_time=START_TIME,
    end_time=END_TIME,
    planting_date=PLANTING_DATE,
    canopy_cover_default=CANOPY_COVER_DEFAULT,
    run_rootzone=RUN_ROOTZONE_PREDICTION,
    target_time=TARGET_TIME,
    anchor_time=ANCHOR_TIME,
    required_history_hours=REQUIRED_HISTORY_HOURS,
    dark_window_hours=DARK_6H_WINDOW_HOURS,
    max_external_gap_hours=MAX_EXTERNAL_GAP_HOURS,
)

print_pipeline_result(result)


ValueError: Anchor row must contain both ph and ec_ms